In [1]:
from build123d import *
from ocp_vscode import *
from math import sin, cos,tan, pi
set_port(3939)
set_defaults(reset_camera=Camera.CENTER, helper_scale=5)
import numpy as np

In [152]:
phone_width=80
phone_depth=15.5
phone_length=154
thick = 3.2
static_angle_d = 80
max_tilt_angle_d = 15

front_notch = 10
back_notch = 25

In [153]:
static_angle_r = static_angle_d * pi/180
max_tilt_angle_r = max_tilt_angle_d * pi/180

a2grav_base = pi /2 - max_tilt_angle_r
a2grav_side = pi - a2grav_base - static_angle_r

base_min_len = phone_length * sin(a2grav_side)/(2 * sin(a2grav_base))
base_min_len

33.68954972355742

In [154]:
base_len = base_min_len + 10

slot_rot = static_angle_d-90
with BuildPart() as part:
    with BuildSketch() as sk:
        Rectangle(base_len, thick, align=Align.MIN)
        Rectangle(thick, back_notch+thick, rotation=slot_rot, align=(Align.MAX,Align.MIN))
        slot_width = phone_depth + thick
        Rectangle(-slot_width, thick, align=(Align.MAX, Align.MIN))
        Rectangle(-slot_width, thick, rotation=slot_rot, align=(Align.MAX, Align.MIN))
        with Locations((-slot_width, 0)):
            Rectangle(thick, front_notch+thick, rotation=slot_rot, align=(Align.MAX,Align.MIN))
        
        
    extrude(amount=phone_width)
    edges = part.edges().group_by(Axis.Z)[1].sort_by(Axis.X)
    o_edges = [*edges[:1], *edges[-6:]]
    edges = edges.sort_by(Axis.Y)
    o_edges.extend(edges[-5:])
    fillet(o_edges, radius=thick/2.1)
    topf =[f for f in part.faces().sort_by(Axis.Y) if f.is_planar][3]
    with BuildSketch(topf):
        Text("Deer", font_size=24, align=Align.CENTER, rotation=90)
    extrude(amount=-thick/4, mode=Mode.SUBTRACT)

show(part)


+


In [155]:
export_stl(part.part, "stand.stl")

True